# 02 - Data Collection with Web Scraping
**IBM Applied Data Science Capstone - SpaceX Falcon 9 First-Stage Landing Prediction**

Author: Pranav Vyankatesh Jagdale

**Goal:** scrape the Falcon 9 launch tables from a fixed Wikipedia revision with BeautifulSoup and build a DataFrame.

> Run every cell in order. Keep your own outputs - they feed your final slides.

In [7]:
import sys
import requests
import re
import unicodedata
import pandas as pd
from bs4 import BeautifulSoup

In [8]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
response = requests.get(static_url, headers={'User-Agent': 'Mozilla/5.0'})
print("status:", response.status_code)
soup = BeautifulSoup(response.text, 'html.parser')
print(soup.title.string)

status: 200
List of Falcon 9 and Falcon Heavy launches - Wikipedia


## Find the launch tables and read the column names

In [9]:
html_tables = soup.find_all('table')
print("tables found:", len(html_tables))
first_launch_table = html_tables[2]   # if this is not the launch table in your run, print other indexes
column_names = []
for th in first_launch_table.find_all('th'):
    text = ' '.join(th.stripped_strings)
    if text and not text.isdigit():
        column_names.append(text)
print(column_names)

tables found: 29
['Flight No.', 'Date and time ( UTC )', 'Version, Booster [ b ]', 'Launch site', 'Payload [ c ]', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome', 'Booster landing']


## Helper functions to clean each cell

In [10]:
def date_time(table_cells):
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    return ''.join([bv for i, bv in enumerate(table_cells.strings) if i % 2 == 0][0:-1])

def landing_status(table_cells):
    return [i for i in table_cells.strings][0]

def get_mass(table_cell):
    mass = unicodedata.normalize("NFKD", table_cell.text).strip()
    if mass:
        return mass[0:mass.find("kg") + 2]
    return 0

def first_text(cell):
    return cell.a.string if cell is not None and cell.a is not None else (cell.get_text(strip=True) if cell is not None else '')

## Parse every launch row

In [11]:
launch_dict = {
    'Flight No.': [], 'Launch site': [], 'Payload': [], 'Payload mass': [], 'Orbit': [],
    'Customer': [], 'Launch outcome': [], 'Version Booster': [], 'Booster landing': [],
    'Date': [], 'Time': [],
}
extracted_row = 0
for table in soup.find_all('table', "wikitable plainrowheaders collapsible"):
    for rows in table.find_all("tr"):
        flag = False
        if rows.th and rows.th.string:
            flight_number = rows.th.string.strip()
            flag = flight_number.isdigit()
        row = rows.find_all('td')
        if flag and len(row) >= 9:
            extracted_row += 1
            launch_dict['Flight No.'].append(flight_number)
            dt = date_time(row[0])
            launch_dict['Date'].append(dt[0].strip(','))
            launch_dict['Time'].append(dt[1] if len(dt) > 1 else '')
            bv = booster_version(row[1])
            if not bv and row[1].a:
                bv = row[1].a.string
            launch_dict['Version Booster'].append(bv)
            launch_dict['Launch site'].append(first_text(row[2]))
            launch_dict['Payload'].append(first_text(row[3]))
            launch_dict['Payload mass'].append(get_mass(row[4]))
            launch_dict['Orbit'].append(first_text(row[5]))
            launch_dict['Customer'].append(first_text(row[6]))
            launch_dict['Launch outcome'].append(list(row[7].strings)[0])
            launch_dict['Booster landing'].append(landing_status(row[8]))
print("rows extracted:", extracted_row)

rows extracted: 121


In [12]:
df = pd.DataFrame({k: pd.Series(v) for k, v in launch_dict.items()})
df.to_csv('my_spacex_web_scraped.csv', index=False)
print(df.shape)
df.head()

(121, 11)


,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Version Booster,Booster landing,Date,Time
0,1,CCAFS,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,F9 v1.07B0003.1,Failure,4 June 2010,18:45
1,2,CCAFS,Dragon,0,LEO,NASA,Success,F9 v1.07B0004.1,Failure,8 December 2010,15:43
2,3,CCAFS,Dragon,525 kg,LEO,NASA,Success,F9 v1.07B0005.1,No,22 May 2012,07:44
3,4,CCAFS,SpaceX CRS-1,"4,700 kg",LEO,NASA,Success,F9 v1.07B0006.1,No attempt,8 October 2012,00:35
4,5,CCAFS,SpaceX CRS-2,"4,877 kg",LEO,NASA,Success,F9 v1.07B0007.1,No,1 March 2013,15:10


## Cross-check with the API data
Compare the number of launches and the landing outcomes here with `my_dataset_part_1.csv` from notebook 01. Write down any differences and why you think they happen (different date ranges, naming, etc.).